In [1]:
import chromadb

client = chromadb.PersistentClient(path="../src/maxidom_hack/db/chroma_db")
collection = client.get_or_create_collection("products")


In [2]:
collection.count()

56076

In [4]:
from dataclasses import dataclass
from typing import Optional
import chromadb

@dataclass
class Retrieved:
    id: str
    text: str
    metadata: dict
    score: float          # cosine similarity = 1 - distance
    distance: float

class ProductRetriever:
    def __init__(self, llm_client, db_path="../src/maxidom_hack/db/chroma_db", name="products"):
        self.llm = llm_client
        self.client = chromadb.PersistentClient(path=db_path)
        self.coll = self.client.get_or_create_collection(name)

    def retrieve(
        self,
        query: str,
        k: int = 5,
        where: Optional[dict] = None,
    ) -> list[Retrieved]:
        q_emb = self.llm.embed_query(query).tolist()
        kwargs = {"query_embeddings": [q_emb], "n_results": k}
        if where:
            kwargs["where"] = where

        res = self.coll.query(**kwargs)

        out = []
        for id_, doc, meta, dist in zip(
            res["ids"][0],
            res["documents"][0],
            res["metadatas"][0],
            res["distances"][0],
        ):
            out.append(Retrieved(
                id=id_, text=doc, metadata=meta,
                score=1.0 - dist, distance=dist,
            ))
        return out

    def context(self, query: str, k: int = 3, where: Optional[dict] = None) -> str:
        """Форматирует retrieved-документы в контекст для LLM."""
        items = self.retrieve(query, k=k, where=where)
        return "\n\n".join(
            f"[{i+1}] {r.metadata.get('category', '')} | "
            f"{r.metadata.get('surface', '')} | "
            f"{r.metadata.get('price', 0):.0f}₽\n{r.text[:300]}"
            for i, r in enumerate(items)
        )

In [5]:
from maxidom_hack.llm_client import LLMClient

llm = LLMClient()

/Users/dolor.src/code/maxidom-hack/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 19:36:18.761000 16539 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 429.94it/s]


model name:  intfloat/multilingual-e5-small
device: mps:0
dtype: torch.float16
max_seq: 256
Реальное устройство модели: mps:0
Устройство первого слоя: mps:0


Loading weights: 100%|██████████| 219/219 [00:00<00:00, 8025.80it/s]


In [6]:
retriever = ProductRetriever(llm)

In [7]:
for q in [
    "керамогранит для пола в ванной",
    "влагостойкая краска для стен",
    "кабель для проводки в квартире",
    "клей для плитки",
    "ламинат для спальни",
]:
    print(f"\n{'='*60}\n{q!r}\n{'='*60}")
    for r in retriever.retrieve(q, k=5):
        print(
            f"  score={r.score:.3f} | "
            f"{r.metadata.get('surface'):10} | "
            f"moist={r.metadata.get('moisture_resistant')} | "
            f"{r.text[:70]!r}"
        )


'керамогранит для пола в ванной'
  score=0.903 | стеновое   | moist=False | 'керамогранит LAPARET Mania 40 2х40,2 серый\nКерамогранит Mania имеет яр'
  score=0.894 | напольное  | moist=False | 'керамогранит LAPARET Атлантик Уайт i 60x60см белый полированный\nКерамо'
  score=0.893 | напольное  | moist=False | 'керамогранит LAPARET Splash Bianco 60х120 белый сатинированный\nКерамог'
  score=0.892 | напольное  | moist=False | 'керамогранит ALMA CERAMICA Ashton 20х90 коричневый матовый\nКерамограни'
  score=0.892 | стеновое   | moist=False | 'керамогранит 32,6x32,6 Light зеленый\nПредназначен для облицовки кварти'

'влагостойкая краска для стен'
  score=0.904 | стеновое   | moist=True | 'краска акриловая ЛАКРА для стен и потолков влагостойкая белоснежная 14'
  score=0.901 | стеновое   | moist=True | 'краска акриловая ЛАКРА для стен и потолков влагостойкая белоснежная 6,'
  score=0.901 | стеновое   | moist=True | 'краска акриловая ЛАКРА для стен и потолков влагостойкая белоснежная 3к'
  sc